# Phase 3 — The debate/judge mechanism

Guardian's core differentiator: instead of silently auto-retraining whenever
performance looks off, four small local LLMs (all via Ollama, `llama3.2:3b`
— nothing leaves this machine) play distinct roles:

1. **Monitor** — looks at a batch of evidence, decides whether it's worth a
   closer look at all. Most batches should NOT be flagged — that's the
   point of having a monitor, rather than debating every single update.
2. **Advocate-for-retrain** — if flagged, argues the strongest honest case
   that this is genuine drift.
3. **Advocate-against-retrain** — argues the strongest honest case that
   it's noise.
4. **Judge** — reviews the evidence and both arguments, and outputs one of
   `auto_approve_retrain`, `auto_reject`, or `escalate_to_human`.

Every step is logged — the full transcript, not just the final verdict —
to `logs/agent_decisions.jsonl`, which is the audit trail Phase 4's
dashboard will read from.

**Design choice:** plain Python calling Ollama directly in sequence, not a
graph framework (LangGraph etc.). This pipeline is linear — monitor, then
maybe debate, then judge — so a framework's main value (state machines,
loops, retries) isn't needed yet, and every step here is a function you can
read top to bottom. See `src/guardian/agents/debate.py`.

In [1]:
import sys
sys.path.insert(0, "../src")

from guardian.agents.audit_log import append_record
from guardian.agents.debate import run_debate
from guardian.agents.scenarios import SCENARIOS, build_context, generate_scenario

ctx = build_context()
print(f"Baseline (from fresh unperturbed simulator draws): RMSE={ctx.baseline_rmse:.2f}, "
      f"imminent-failure rate={ctx.baseline_imminent_rate:.1%}")

/Users/meetadave/Desktop/Projects/guardian/.venv/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000486 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2824
[LightGBM] [Info] Number of data points in the train set: 17400, number of used features: 16
[LightGBM] [Info] Start training from score 86.530172


Baseline (from fresh unperturbed simulator draws): RMSE=30.29, imminent-failure rate=10.4%


## The evidence packet

Two complementary signals feed the agents, both computable without knowing
a currently-running engine's future:

- **Retrospective performance** — for a batch of engines that have SINCE
  reached failure, how did the model's RUL predictions compare to what
  actually happened, across every recorded cycle (not just one point per
  engine — see below for why that matters).
- **Live sensor drift** — does this batch's early-life ("healthy") sensor
  readings look like what the Phase 2 simulator calibrated from training
  data, or a new regime? Doesn't need any engine to have failed yet.
- **Digital-twin comparison** — reuses the Phase 2 simulator's fitted
  per-engine decline-rate distribution: how does this batch's implied
  degradation rate compare to the 100 real engines used to calibrate it?

### A real finding while building this: GBM's RMSE can be a misleading drift signal

Tree models can't extrapolate past their training range. A sensor value far
outside anything seen in training collapses into whatever leaf sits at the
edge of the tree — which can look stable, or even *improve* the RMSE
number, even though the model has no real understanding of what it's
looking at. This was observed directly: a batch with sensor_4/sensor_11
shifted ~4 SD outside the calibrated range showed RMSE *improving* by over
20% versus baseline, not degrading. Chasing this confirmed it's systematic,
not a fluke — averaging across every rate/shift combination tested, GBM's
RMSE consistently looks flat-to-better under the exact kinds of covariate
shift that should worry a monitoring system.

That's why the evidence includes signals that don't depend on the model's
own predictions at all (sensor drift, decline-rate comparison), and why
`Evidence.rmse_may_be_misleading` fires an explicit caveat in the prompt
text whenever sensor drift is large but RMSE looks fine — telling the
agents plainly not to trust RMSE alone in that case. See
`src/guardian/agents/evidence.py`.

## Three scenarios, with known ground truth

Built from the Phase 2 simulator so we know what "should" happen, which
lets us sanity-check the mechanism rather than just trust it blindly:

- **noise**: sampled from the simulator's own calibrated distribution, no
  injected shift. Expect: not flagged, or flagged-then-rejected.
- **genuine_drift**: a faster decline rate *plus* a sensor shift outside
  the simulator's own fitted family (standing in for e.g. a sensor
  recalibration or new fault mode). Expect: flagged, debated, and — given
  how deceptive the RMSE signal is here — a real test of whether the
  agents correctly lean on the other evidence instead.
- **ambiguous**: a small batch (n=6, not 30) with a modest rate increase —
  enough signal to be worth a look, too little (and too few samples) to be
  conclusive. Expect: a genuinely harder call.

In [2]:
results = {}
for scenario in SCENARIOS:
    evidence = generate_scenario(ctx, scenario)
    result = run_debate(evidence)
    append_record(result, model="llama3.2:3b")
    results[scenario] = result
    print(f"[{scenario}] flagged={result.monitor_flag}  judge={result.judge_decision}")

[noise] flagged=False  judge=None


[genuine_drift] flagged=True  judge=escalate_to_human


[ambiguous] flagged=False  judge=None


## Scenario 1: noise

In [3]:
print(results["noise"].evidence.to_prompt_text())
print()
print("MONITOR:", results["noise"].monitor_flag)
print(" ", results["noise"].monitor_reasoning)

BATCH: noise (30 engines with completed run-to-failure data)

MODEL PERFORMANCE (retrospective, now that true failure times are known):
- Historical validation RMSE: 30.29 cycles
- This batch's RMSE: 29.09 cycles (-4.0% vs. historical)
- Historical rate of "imminent failure" cycles (RUL<=20) in the data: 10.4%
- This batch's rate: 10.8%

SENSOR DRIFT (early-life "healthy" readings vs. training baseline, in standard deviations):
- sensor_15: -0.5 SD
- sensor_2: -0.5 SD
- sensor_21: +0.4 SD

DIGITAL-TWIN COMPARISON:
- This batch's engines degrade at 0.95x the typical calibrated rate (1.0x = average).
- Historical percentile: 56th out of 100 real calibration engines, ranked from slowest- to fastest-degrading. Verdict: unremarkable — squarely in the middle of normal historical variation.

MONITOR: False
  While the model performance shows a slight improvement, the sensor readings are within 0.5 standard deviations of the training baseline, indicating no significant drift. The digital-twin 

**Outcome: correctly not flagged.** Every signal (RMSE, sensor drift,
decline-rate percentile) reads as unremarkable, and the Monitor declined to
escalate — no debate needed, no compute spent arguing about nothing. This
is exactly the common-case behavior a monitor should have: most incoming
data isn't interesting, and the whole point of the Monitor stage is to
filter that out before it reaches the more expensive debate step.

## Scenario 2: genuine_drift

In [4]:
r = results["genuine_drift"]
print(r.evidence.to_prompt_text())
print()
print("MONITOR:", r.monitor_flag, "\n ", r.monitor_reasoning)
print("\nADVOCATE-FOR:\n ", r.advocate_for_text)
print("\nADVOCATE-AGAINST:\n ", r.advocate_against_text)
print("\nJUDGE:", r.judge_decision, f"(confidence: {r.judge_confidence})")
print(" ", r.judge_rationale)

BATCH: genuine_drift (30 engines with completed run-to-failure data)

MODEL PERFORMANCE (retrospective, now that true failure times are known):
- Historical validation RMSE: 30.29 cycles
- This batch's RMSE: 24.01 cycles (-20.7% vs. historical)
- Historical rate of "imminent failure" cycles (RUL<=20) in the data: 10.4%
- This batch's rate: 9.4%

SENSOR DRIFT (early-life "healthy" readings vs. training baseline, in standard deviations):
- sensor_11: -4.5 SD
- sensor_4: +4.1 SD
- sensor_7: +0.3 SD
- CAVEAT: sensor drift here is large enough that RMSE may be an unreliable signal on its own. Tree-based models cannot extrapolate past their training range, so a sensor value far outside anything seen in training can look stable (or even improve) in the RMSE number even when the model is really just extrapolating blindly. Weigh the sensor drift and digital-twin comparison below more heavily than RMSE in this case.

DIGITAL-TWIN COMPARISON:
- This batch's engines degrade at 1.55x the typical ca

**Outcome: flagged, fully debated, escalated to human at low confidence
— not the clean "auto-approve" we'd naively hypothesize from the ground
truth (we know we injected a real shift).** This is worth sitting with
rather than treating as a failure.

The Monitor correctly flagged it, citing the 87th-percentile decline rate
and the large sensor drift. Both advocates produced coherent,
evidence-grounded arguments — Advocate-for leaned on the sensor drift and
the caveat about RMSE reliability; Advocate-against leaned on the
"improved" RMSE and the decline rate being "not extreme." The Judge, faced
with a genuinely contradictory RMSE signal even with an explicit warning
about it, chose the cautious path.

Two honest readings of this:
1. **It's the intended fail-safe working correctly.** Guardian's whole
   design principle is that genuinely mixed evidence should go to a human,
   not force a confident-sounding guess. A 3B local model facing evidence
   where one number says "better" and three others say "worse" choosing to
   punt is arguably the *safe* behavior, not a bug.
2. **It's also a real limit of a 3B model's reasoning under adversarial
   framing.** The caveat text explicitly told the Judge to weight sensor
   drift over RMSE here, and it still hedged. A larger model (we have
   `qwen2.5:7b-instruct` available, just impractically slow on this
   machine — see Phase 3's README section) might resolve this more
   decisively. That's a legitimate lever for a future iteration, not
   something to paper over here.

Both readings matter for the audit story this project is built around:
the transcript shows *exactly* why the system landed where it did, which
is the whole point of debate agents over a black-box classifier.

## Scenario 3: ambiguous

In [5]:
r = results["ambiguous"]
print(r.evidence.to_prompt_text())
print()
print("MONITOR:", r.monitor_flag, "\n ", r.monitor_reasoning)

BATCH: ambiguous (6 engines with completed run-to-failure data)

MODEL PERFORMANCE (retrospective, now that true failure times are known):
- Historical validation RMSE: 30.29 cycles
- This batch's RMSE: 27.43 cycles (-9.5% vs. historical)
- Historical rate of "imminent failure" cycles (RUL<=20) in the data: 10.4%
- This batch's rate: 10.9%

SENSOR DRIFT (early-life "healthy" readings vs. training baseline, in standard deviations):
- sensor_15: -1.1 SD
- sensor_4: -1.0 SD
- sensor_8: -1.0 SD

DIGITAL-TWIN COMPARISON:
- This batch's engines degrade at 1.03x the typical calibrated rate (1.0x = average).
- Historical percentile: 62nd out of 100 real calibration engines, ranked from slowest- to fastest-degrading. Verdict: unremarkable — squarely in the middle of normal historical variation.

MONITOR: False 
  While the model performance and sensor readings show some minor improvements, the overall trends are within normal historical variation. The slight increase in 'imminent failure' rate 

**Outcome: not flagged.** With only 6 engines and a modest 1.03x decline
rate (62nd percentile — squarely unremarkable), the Monitor reasonably
judged this within normal variation. In hindsight, this scenario's injected
signal (rate_boost=1.3, no sensor shift) turned out to be milder than
"noise" needs to be dramatic *and* uncertain to really test the
escalate-to-human path — `genuine_drift` ended up being the scenario that
did that instead, just via debate rather than the Monitor gate. A sharper
"ambiguous" scenario (strong enough to flag, genuinely 50/50 once debated)
is a good candidate for follow-up scenario design, tracked rather than
silently fixed by cherry-picking parameters until the "expected" label
appears.

## Summary

| Scenario | Monitor flagged? | Debate ran? | Judge decision |
|---|---|---|---|
| noise | No | No | — |
| genuine_drift | Yes | Yes | escalate_to_human (low confidence) |
| ambiguous | No | No | — |

Two of three scenarios never needed the expensive debate step at all —
the Monitor correctly filtered them as unremarkable, which is the system
working as designed, not the system failing to find something interesting.
The one that did get debated produced a full, evidence-grounded transcript
and a cautious-but-defensible verdict, logged in full to
`logs/agent_decisions.jsonl` for audit.

**What this sets up for Phase 4:** the audit log already has everything a
dashboard needs — evidence snapshots, both arguments, the verdict, parse
errors, and per-call timings — one JSON object per decision, ready for
`pandas.read_json(path, lines=True)`.